# 02 Split and Verify Dataset

This notebook creates the original train/validation/test split for factory sign detection and verifies the split quality before any augmentation or training happens.


## Purpose

Notebook 02 has two jobs: copy valid input image-label pairs into deterministic train/val/test folders, then run post-split EDA to check class balance, no-sign distribution, bbox size/location, image quality, and exact duplicate leakage.

The test split is inspected here only for split-quality verification. Do not use these observations to select architecture, tune augmentation strength, choose hyperparameters, or tune confidence thresholds.


## 1. Setup

Find the `sign-detection` root, add it to `sys.path`, and import the split, EDA, and plotting helpers.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


# Notebook execution can start from the repository root or from notebooks/.
# This helper walks upward until it finds the sign-detection project layout.
def find_project_root(start: Path) -> Path:
    """Return the sign-detection project root from a notebook working directory."""
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "class_names.yaml").exists() and (
            candidate / "src"
        ).exists():
            return candidate
        sign_child = candidate / "sign-detection"
        if (sign_child / "configs" / "class_names.yaml").exists() and (
            sign_child / "src"
        ).exists():
            return sign_child
    raise RuntimeError("Could not find sign-detection project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# Add the project root so imports like src.dataset.split_dataset work in Jupyter.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Keep the notebook thin: reusable split, EDA, and plotting logic lives in src/.
from src.dataset.split_dataset import split_dataset
from src.analysis.dataset_eda import (
    attach_split_to_bbox_records,
    build_split_warnings,
    check_exact_duplicate_leakage,
    summarize_bbox_statistics_by_split,
    summarize_image_quality_by_split,
    summarize_split_class_distribution,
    summarize_split_distribution,
    summarize_split_image_presence,
    summarize_split_no_sign_distribution,
)
from src.analysis.plot_utils import (
    plot_bbox_area_by_class,
    plot_bbox_center_heatmap,
    plot_image_quality_by_split,
    plot_sample_annotations_grid,
    plot_split_class_distribution,
    plot_split_no_sign_ratio,
)

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Github\smart-factory-safety-monitoring-system\sign-detection


## 2. Load Configuration and Notebook 01 Profile Outputs

Notebook 02 uses `input_image_profile.csv` for splitting and `input_bbox_records.csv` for post-split bbox analysis. If these files are missing, run Notebook 01 first.


In [ ]:
# Load class names and dataset paths from config so the notebook has no hardcoded absolute paths.
with (PROJECT_ROOT / "configs" / "class_names.yaml").open(
    "r", encoding="utf-8"
) as file:
    class_config = yaml.safe_load(file)
with (PROJECT_ROOT / "configs" / "dataset_config.yaml").open(
    "r", encoding="utf-8"
) as file:
    dataset_config = yaml.safe_load(file)

# Convert YAML keys to integers because pandas/Ultralytics expect numeric class IDs.
class_names = {int(class_id): name for class_id, name in class_config["names"].items()}
paths = dataset_config["paths"]

# Source folders are read-only inputs; generated splits and reports are written elsewhere.
input_images_dir = PROJECT_ROOT / paths["input_images"]
input_labels_dir = PROJECT_ROOT / paths["input_labels"]
output_splits_dir = PROJECT_ROOT / paths["splits_original"]
profile_dir = PROJECT_ROOT / "reports" / "profile"
splits_report_dir = PROJECT_ROOT / "reports" / "splits"
eda_report_dir = PROJECT_ROOT / paths.get("reports_eda", "reports/eda")
figures_dir = PROJECT_ROOT / paths.get("reports_figures", "reports/figures")

# Notebook 01 creates these profile files. Notebook 02 should stop early if they are missing.
image_profile_path = profile_dir / "input_image_profile.csv"
bbox_records_path = profile_dir / "input_bbox_records.csv"
if not image_profile_path.exists():
    raise FileNotFoundError(f"Missing {image_profile_path}. Run Notebook 01 first.")
if not bbox_records_path.exists():
    raise FileNotFoundError(f"Missing {bbox_records_path}. Run Notebook 01 first.")

image_profile_df = pd.read_csv(image_profile_path)
bbox_records_df = pd.read_csv(bbox_records_path)

# Report folders are generated artifacts; they are safe to create when the notebook runs.
splits_report_dir.mkdir(parents=True, exist_ok=True)
eda_report_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Loaded image profile rows: {len(image_profile_df)}")
print(f"Loaded bbox records: {len(bbox_records_df)}")


Loaded image profile rows: 475
Loaded bbox records: 514


## 3. Review Intended Split Ratios

The split ratios and seed come from `configs/dataset_config.yaml`. Set `OVERWRITE_SPLITS = True` only when you intentionally want to regenerate existing split folders.


In [ ]:
# Split ratios and seed come from dataset_config.yaml for reproducibility.
split_config = dataset_config["split"]
train_ratio = float(split_config["train"])
val_ratio = float(split_config["val"])
test_ratio = float(split_config["test"])
random_seed = int(dataset_config.get("random_seed", 42))

# Safety switch: keep False for normal use. Set True only when deliberately regenerating splits.
OVERWRITE_SPLITS = False

print("Train ratio:", train_ratio)
print("Val ratio:", val_ratio)
print("Test ratio:", test_ratio)
print("Random seed:", random_seed)
print("Overwrite existing splits:", OVERWRITE_SPLITS)

# This preview shows how many images belong to each stratification group.
display(
    image_profile_df["split_group_key"]
    .value_counts()
    .rename_axis("split_group_key")
    .reset_index(name="num_images")
)


Train ratio: 0.7
Val ratio: 0.2
Test ratio: 0.1
Random seed: 42
Overwrite existing splits: False


,split_group_key,num_images
0,M014=1|M015=0|P004=0|W011=0|NONE=0,115
1,M014=0|M015=0|P004=0|W011=1|NONE=0,109
2,M014=0|M015=0|P004=1|W011=0|NONE=0,108
3,M014=0|M015=1|P004=0|W011=0|NONE=0,102
4,M014=0|M015=0|P004=0|W011=0|NONE=1,30
5,M014=1|M015=1|P004=1|W011=1|NONE=0,11


## 4. Create Train/Val/Test Split

The splitter first tries stratification by `split_group_key`. If groups are too small, it falls back to deterministic random splitting and records that in the assignment notes.


In [ ]:
# Copy valid image-label pairs into generated train/val/test folders.
# The helper refuses to overwrite existing split files unless OVERWRITE_SPLITS=True.
split_df = split_dataset(
    image_profile_df=image_profile_df,
    input_images_dir=input_images_dir,
    input_labels_dir=input_labels_dir,
    output_splits_dir=output_splits_dir,
    train_ratio=train_ratio,
    val_ratio=val_ratio,
    test_ratio=test_ratio,
    seed=random_seed,
    overwrite=OVERWRITE_SPLITS,
)

# Save one row per image so later notebooks can trace exactly where each sample went.
split_assignments_path = splits_report_dir / "split_assignments.csv"
split_df.to_csv(split_assignments_path, index=False)

print(f"Saved split assignments: {split_assignments_path}")
print("Split notes:")
for note in split_df["notes"].dropna().astype(str).unique():
    print("-", note)

# Show a compact sample of assignments for quick manual review.
display(
    split_df[
        ["image_name", "split", "split_group_key", "is_no_sign", "status", "notes"]
    ].head(12)
)


Saved split assignments: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\splits\split_assignments.csv
Split notes:
- 


,image_name,split,split_group_key,is_no_sign,status,notes
0,M014_002.png,test,M014=1|M015=0|P004=0|W011=0|NONE=0,False,copied,
1,M014_010.png,test,M014=1|M015=0|P004=0|W011=0|NONE=0,False,copied,
2,M014_017.jpg,test,M014=1|M015=0|P004=0|W011=0|NONE=0,False,copied,
3,M014_039.png,test,M014=1|M015=0|P004=0|W011=0|NONE=0,False,copied,
4,M014_041.png,test,M014=1|M015=0|P004=0|W011=0|NONE=0,False,copied,
5,M014_059.png,test,M014=1|M015=0|P004=0|W011=0|NONE=0,False,copied,
6,M014_080.png,test,M014=1|M015=0|P004=0|W011=0|NONE=0,False,copied,
7,M014_083.png,test,M014=1|M015=0|P004=0|W011=0|NONE=0,False,copied,
8,M014_086.png,test,M014=1|M015=0|P004=0|W011=0|NONE=0,False,copied,
9,M014_097.png,test,M014=1|M015=0|P004=0|W011=0|NONE=0,False,copied,


## 5. Save YOLO Dataset YAML

This YAML points Ultralytics at the generated original split. Paths are relative to `sign-detection`.


In [5]:
# Ultralytics expects dataset YAML paths to be relative to the YAML file location.
# This file describes only the original split; augmentation and ablations come later.
yolo_yaml_path = PROJECT_ROOT / "data_splits_original.yaml"
yolo_yaml = {
    "path": "data/generated/splits_original",
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": int(class_config["nc"]),
    "names": class_names,
}
with yolo_yaml_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(yolo_yaml, file, sort_keys=False)
print(f"Saved YOLO dataset YAML: {yolo_yaml_path}")


Saved YOLO dataset YAML: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data_splits_original.yaml


## 6. Build Post-Split EDA Tables

These reports verify whether train, validation, and test are balanced enough before moving to train-only offline augmentation.


In [ ]:
# Build split-level tables for image counts, class balance, no-sign balance, and bbox statistics.
split_summary_df = summarize_split_distribution(split_df)
split_class_distribution_df = summarize_split_class_distribution(
    split_df, class_names=class_names
)
split_image_presence_df = summarize_split_image_presence(
    split_df, class_names=class_names
)
split_no_sign_df = summarize_split_no_sign_distribution(split_df)

# Attach each original bbox record to its assigned split for object-size/location analysis.
bbox_with_split_df = attach_split_to_bbox_records(bbox_records_df, split_df)
split_bbox_stats_df = summarize_bbox_statistics_by_split(
    bbox_with_split_df, class_names=class_names
)

# These checks use copied split images so they reflect the dataset YOLO will train/evaluate on.
split_image_quality_df = summarize_image_quality_by_split(split_df)
split_duplicate_check_df = check_exact_duplicate_leakage(split_df)
split_warnings_df = build_split_warnings(split_df, bbox_with_split_df)

# Save compact reports. These are generated artifacts and may be overwritten when this cell reruns.
split_summary_df.to_csv(splits_report_dir / "split_summary.csv", index=False)
split_warnings_df.to_csv(splits_report_dir / "split_warnings.csv", index=False)
split_class_distribution_df.to_csv(
    eda_report_dir / "split_class_distribution.csv", index=False
)
split_image_presence_df.to_csv(
    eda_report_dir / "split_image_presence_distribution.csv", index=False
)
split_no_sign_df.to_csv(eda_report_dir / "split_no_sign_distribution.csv", index=False)
split_bbox_stats_df.to_csv(eda_report_dir / "split_bbox_statistics.csv", index=False)
split_image_quality_df.to_csv(
    eda_report_dir / "split_image_quality_summary.csv", index=False
)
split_duplicate_check_df.to_csv(
    eda_report_dir / "split_duplicate_check.csv", index=False
)

print("Saved split and EDA reports.")


Saved split and EDA reports.


## 7. Review Split Summary Tables

Start with the compact split overview, then inspect object-level class balance, image-level class presence, and no-sign distribution.


In [7]:
# These tables answer the first split-quality questions: size, labels, no-sign balance, and class balance.
print("Split overview")
display(split_summary_df)

print("Object-level class distribution")
display(split_class_distribution_df)

print("Image-level class presence")
display(split_image_presence_df)

print("No-sign distribution")
display(split_no_sign_df)


Split overview


,split,num_images,num_labels,num_labeled_images,num_no_sign_images,no_sign_ratio,total_objects
0,test,47,47,44,3,0.063830,48
1,train,332,332,311,21,0.063253,366
2,val,96,96,90,6,0.062500,100


Object-level class distribution


,split,class_id,class_name,object_count,object_ratio_within_split
0,test,0,M014_Helmet,12,0.250000
1,test,1,M015_Vest,11,0.229167
2,test,2,P004_NoThoroughfare,12,0.250000
3,test,3,W011_Slippery,13,0.270833
4,train,0,M014_Helmet,92,0.251366
5,train,1,M015_Vest,90,0.245902
6,train,2,P004_NoThoroughfare,94,0.256831
7,train,3,W011_Slippery,90,0.245902
8,val,0,M014_Helmet,25,0.250000
9,val,1,M015_Vest,25,0.250000


Image-level class presence


,split,class_id,class_name,images_with_class,image_presence_ratio
0,test,0,M014_Helmet,12,0.255319
1,test,1,M015_Vest,11,0.234043
2,test,2,P004_NoThoroughfare,12,0.255319
3,test,3,W011_Slippery,12,0.255319
4,train,0,M014_Helmet,89,0.268072
5,train,1,M015_Vest,79,0.237952
6,train,2,P004_NoThoroughfare,83,0.250000
7,train,3,W011_Slippery,84,0.253012
8,val,0,M014_Helmet,25,0.260417
9,val,1,M015_Vest,23,0.239583


No-sign distribution


,split,num_no_sign_images,no_sign_ratio
0,test,3,0.063830
1,train,21,0.063253
2,val,6,0.062500


## 8. Review BBox Size, Image Quality, and Duplicate Leakage

Small signs are expected in CCTV footage, but extreme imbalance or duplicate leakage across splits should be reviewed before augmentation.


In [8]:
# Bbox statistics highlight tiny signs, which are common and important for CCTV-style detection.
print("Bounding-box statistics")
display(split_bbox_stats_df)

# Brightness and blur summaries help catch accidental split imbalance in image quality.
print("Image quality summary")
display(split_image_quality_df)

# Exact duplicate leakage means identical image content appears in more than one split.
print("Exact duplicate leakage across splits")
if split_duplicate_check_df.empty:
    print("No exact duplicate leakage found.")
else:
    display(split_duplicate_check_df)


Bounding-box statistics


,split,class_id,class_name,num_boxes,mean_box_area_norm,median_box_area_norm,mean_box_width_px,mean_box_height_px,tiny_box_count,tiny_box_ratio,small_box_count,small_box_ratio
0,test,0,M014_Helmet,12,0.056732,0.030366,273.834646,284.371806,0,0.000000,0,0.000000
1,test,1,M015_Vest,11,0.053765,0.005209,158.309085,171.052854,0,0.000000,0,0.000000
2,test,2,P004_NoThoroughfare,12,0.032354,0.012028,209.184782,222.688159,0,0.000000,0,0.000000
3,test,3,W011_Slippery,13,0.059469,0.013417,302.284475,296.652249,0,0.000000,0,0.000000
4,train,0,M014_Helmet,92,0.069463,0.027335,306.915624,339.142079,0,0.000000,5,0.054348
5,train,1,M015_Vest,90,0.022693,0.007799,137.675381,150.115308,0,0.000000,8,0.088889
6,train,2,P004_NoThoroughfare,94,0.022748,0.008850,159.671288,182.480013,1,0.010638,5,0.053191
7,train,3,W011_Slippery,90,0.020881,0.009686,215.956867,212.940532,0,0.000000,8,0.088889
8,val,0,M014_Helmet,25,0.099867,0.047759,385.561197,406.591638,0,0.000000,0,0.000000
9,val,1,M015_Vest,25,0.026947,0.007639,130.171057,155.450464,0,0.000000,3,0.120000


Image quality summary


,split,num_images,mean_image_width,mean_image_height,mean_aspect_ratio,mean_brightness_mean,mean_brightness_std,mean_blur_score_laplacian,median_blur_score_laplacian
0,test,47,1871.085106,1197.340426,1.608941,99.842787,46.885910,914.451456,436.946594
1,train,332,1910.647590,1216.584337,1.609230,97.440263,45.944289,707.280691,365.428940
2,val,96,1889.072917,1181.625000,1.627856,93.600893,45.715621,723.715569,371.361542


Exact duplicate leakage across splits
No exact duplicate leakage found.


## 9. Generate Verification Figures

A small set of figures is enough here: class balance, no-sign ratio, bbox area, bbox location, image quality, and one sample annotation grid.


In [ ]:
# Generate only the core figures needed for split review; avoid a pile of noisy artifacts.
figure_jobs = [
    (
        plot_split_class_distribution,
        split_class_distribution_df,
        figures_dir / "split_class_distribution.png",
    ),
    (
        plot_split_no_sign_ratio,
        split_no_sign_df,
        figures_dir / "split_no_sign_ratio.png",
    ),
    (
        plot_bbox_area_by_class,
        bbox_with_split_df,
        figures_dir / "bbox_area_by_class.png",
    ),
    (
        plot_bbox_center_heatmap,
        bbox_with_split_df,
        figures_dir / "bbox_center_heatmap.png",
    ),
    (
        plot_image_quality_by_split,
        split_image_quality_df,
        figures_dir / "image_quality_by_split.png",
    ),
]
for plot_func, data, output_path in figure_jobs:
    try:
        plot_func(data, output_path)
        print(f"Saved {output_path}")
    except Exception as exc:
        print(f"Skipped {output_path.name}: {exc}")

# One mixed grid is enough for visual sanity: labeled, no-sign, and tiny-sign examples when available.
try:
    sample_grid_path = plot_sample_annotations_grid(
        split_df=split_df,
        bbox_with_split=bbox_with_split_df,
        output_path=figures_dir / "sample_annotations_grid.png",
    )
    print(f"Saved {sample_grid_path}")
except Exception as exc:
    print(f"Skipped sample annotation grid: {exc}")


Saved C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\figures\split_class_distribution.png
Saved C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\figures\split_no_sign_ratio.png
Saved C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\figures\bbox_area_by_class.png
Saved C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\figures\bbox_center_heatmap.png
Saved C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\figures\image_quality_by_split.png
Saved C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\figures\sample_annotations_grid.png


## 10. Review Warnings

Warnings are review prompts, not automatic blockers. Fix the dataset or rerun with a different seed only when a warning reveals a real split-quality problem.


In [10]:
# Warnings are review prompts. They do not automatically mean the split is unusable.
if split_warnings_df.empty:
    print("No split warnings generated.")
else:
    display(split_warnings_df)


No split warnings generated.


## 11. Notebook 03 Checklist

Before moving to Notebook 03:

- Confirm `reports/splits/split_assignments.csv` exists.
- Confirm `data/generated/splits_original/` has train, val, and test image-label pairs.
- Review class balance and image-level class presence.
- Review no-sign distribution across splits.
- Review tiny/small bbox statistics.
- Confirm no exact duplicate leakage crosses split boundaries.
- Keep validation and test untouched after this point; Notebook 03 should augment training images only.
